# 12 - Product Report

This notebook creates a reusable product-level analytical report view.

The report combines product attributes with sales behavior and calculates:
- product category and subcategory
- total orders
- total sales
- total quantity sold
- unique customers
- product lifespan
- recency since last sale
- average selling price
- average order revenue
- average monthly revenue
- product performance segment

In [0]:
%sql
/*
Product Report View
--------------------

Purpose:
    Create a reusable product-level analytical view by combining product
    attributes with sales behavior.

Output:
    One row per product with product attributes, sales metrics,
    lifecycle metrics, and performance segmentation.
*/

CREATE OR REPLACE VIEW datawarehouseanalytics_gold.report_products AS

WITH base_query AS (
    SELECT
        f.order_number,
        f.order_date,
        f.customer_key,
        f.sales_amount,
        f.quantity,
        p.product_key,
        p.product_name,
        COALESCE(p.category, 'Unknown') AS category,
        COALESCE(p.subcategory, 'Unknown') AS subcategory,
        p.cost
    FROM datawarehouseanalytics_gold.fact_sales f
    LEFT JOIN datawarehouseanalytics_gold.dim_products p
        ON f.product_key = p.product_key
    WHERE f.order_date IS NOT NULL
),

product_aggregations AS (
    SELECT
        product_key,
        product_name,
        category,
        subcategory,
        cost,

        MIN(order_date) AS first_sale_date,
        MAX(order_date) AS last_sale_date,
        ROUND(MONTHS_BETWEEN(MAX(order_date), MIN(order_date)), 0) AS lifespan_months,

        COUNT(DISTINCT order_number) AS total_orders,
        COUNT(DISTINCT customer_key) AS total_customers,
        SUM(sales_amount) AS total_sales,
        SUM(quantity) AS total_quantity,

        ROUND(
            AVG(CASE 
                    WHEN quantity = 0 THEN NULL
                    ELSE sales_amount / quantity
                END),
            2
        ) AS avg_selling_price

    FROM base_query
    GROUP BY
        product_key,
        product_name,
        category,
        subcategory,
        cost
)

SELECT
    product_key,
    product_name,
    category,
    subcategory,
    cost,

    first_sale_date,
    last_sale_date,

    ROUND(MONTHS_BETWEEN(CURRENT_DATE(), last_sale_date), 0) AS recency_months,

    CASE
        WHEN total_sales > 50000 THEN 'High Performer'
        WHEN total_sales >= 10000 THEN 'Mid Range'
        ELSE 'Low Performer'
    END AS product_segment,

    lifespan_months,
    total_orders,
    total_sales,
    total_quantity,
    total_customers,
    avg_selling_price,

    CASE 
        WHEN total_orders = 0 THEN 0
        ELSE ROUND(total_sales / total_orders, 2)
    END AS avg_order_revenue,

    CASE
        WHEN lifespan_months = 0 THEN total_sales
        ELSE ROUND(total_sales / lifespan_months, 2)
    END AS avg_monthly_revenue

FROM product_aggregations;

In [0]:
%sql
SELECT *
FROM datawarehouseanalytics_gold.report_products
LIMIT 10;

In [0]:
%sql
SELECT COUNT(*) AS total_products_in_report
FROM datawarehouseanalytics_gold.report_products;

In [0]:
%sql
SELECT
    product_segment,
    COUNT(*) AS total_products,
    SUM(total_sales) AS total_sales,
    SUM(total_quantity) AS total_quantity,
    ROUND(AVG(avg_selling_price), 2) AS avg_selling_price,
    ROUND(AVG(avg_order_revenue), 2) AS avg_order_revenue
FROM datawarehouseanalytics_gold.report_products
GROUP BY product_segment
ORDER BY total_sales DESC;